In [1]:
import os
print(os.getcwd())

/Users/thanadolpoonsawat/Desktop/project/Big data compute/src


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Original local-style SparkSession config kept for reference.
# spark = SparkSession.builder \
#     .appName("fraud") \
#     .config("spark.driver.memory", "8g") \
#     .config("spark.executor.memory", "8g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .getOrCreate()

GCS_BASE = "gs://bigdata-2603572-team3-storge/big data project/Big data compute"
RAW_DATA_PATH = f"{GCS_BASE}/data/raw data"
FEATURE_DATA_PATH = f"{GCS_BASE}/data/feature"

# Tuned for a small Dataproc/JupyterLab cluster with 2 workers.
# Keep this lightweight like the archived notebook: no local master, no hard-coded memory.
SHUFFLE_PARTITIONS = 32

spark = (
    SparkSession.builder
    .appName("fraud-feature-engineering")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.eventLog.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Event log enabled:", spark.conf.get("spark.eventLog.enabled"))
print("GCS base:", GCS_BASE)
print("Raw data path:", RAW_DATA_PATH)
print("Feature data path:", FEATURE_DATA_PATH)


In [ ]:
users = spark.read.option("header", True).csv(f"{RAW_DATA_PATH}/users.csv")
cards = spark.read.option("header", True).csv(f"{RAW_DATA_PATH}/cards.csv")
transactions = spark.read.option("header", True).csv(f"{RAW_DATA_PATH}/transactions.csv")
print("users columns:", len(users.columns))
print("cards columns:", len(cards.columns))
print("transactions columns:", len(transactions.columns))


In [4]:
users = users \
    .withColumn("user", F.col("user").cast("int")) \
    .withColumn("current_age", F.col("current_age").cast("int")) \
    .withColumn("retirement_age", F.col("retirement_age").cast("int")) \
    .withColumn("birth_year", F.col("birth_year").cast("int")) \
    .withColumn("birth_month", F.col("birth_month").cast("int")) \
    .withColumn("apartment", F.col("apartment").cast("double")) \
    .withColumn("zipcode", F.col("zipcode").cast("int")) \
    .withColumn("latitude", F.col("latitude").cast("double")) \
    .withColumn("longitude", F.col("longitude").cast("double")) \
    .withColumn("per_capita_income_zipcode", F.col("per_capita_income_zipcode").cast("double")) \
    .withColumn("yearly_income_person", F.col("yearly_income_person").cast("double")) \
    .withColumn("total_debt", F.col("total_debt").cast("double")) \
    .withColumn("fico_score", F.col("fico_score").cast("int")) \
    .withColumn("num_credit_cards", F.col("num_credit_cards").cast("int"))

cards = cards \
    .withColumn("user", F.col("user").cast("int")) \
    .withColumn("card", F.col("card").cast("int")) \
    .withColumn("card_number", F.col("card_number").cast("long")) \
    .withColumn("cvv", F.col("cvv").cast("int")) \
    .withColumn("cards_issued", F.col("cards_issued").cast("int")) \
    .withColumn("credit_limit", F.col("credit_limit").cast("double")) \
    .withColumn("year_pin_last_changed", F.col("year_pin_last_changed").cast("int"))

transactions = transactions \
    .withColumn("user", F.col("user").cast("int")) \
    .withColumn("card", F.col("card").cast("int")) \
    .withColumn("year", F.col("year").cast("int")) \
    .withColumn("month", F.col("month").cast("int")) \
    .withColumn("day", F.col("day").cast("int")) \
    .withColumn("amount", F.col("amount").cast("double")) \
    .withColumn("merchant_name", F.col("merchant_name").cast("long")) \
    .withColumn("zip", F.col("zip").cast("double").cast("int")) \
    .withColumn("mcc", F.col("mcc").cast("int"))

# user feature

In [5]:
user_features = users.select(
    "user",
    "current_age",
    "retirement_age",
    "state",
    "zipcode",
    "latitude",
    "longitude",
    "yearly_income_person",
    "total_debt",
    "fico_score",
    "num_credit_cards",
    "apartment"
).withColumn(
    "years_to_retirement",
    F.col("retirement_age") - F.col("current_age")
).withColumn(
    "debt_to_income_ratio",
    F.col("total_debt") / (F.col("yearly_income_person") + F.lit(1.0))
).withColumn(
    "income_per_card",
    F.col("yearly_income_person") / (F.col("num_credit_cards") + F.lit(1.0))
).withColumn(
    "apartment_missing",
    F.when(F.col("apartment").isNull(), 1).otherwise(0)
).drop("apartment")

In [ ]:
user_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/user_features.parquet")

# card feature

In [7]:
cards = cards \
    .withColumn("acct_open_date_parsed", F.to_date(F.concat(F.lit("01/"), F.col("acct_open_date")), "dd/MM/yyyy")) \
    .withColumn("expires_parsed", F.to_date(F.concat(F.lit("01/"), F.col("expires")), "dd/MM/yyyy"))

ref_date = F.to_date(F.lit("2020-02-29"))

card_features = cards.select(
    "user",
    "card",
    "card_brand",
    "card_type",
    "has_chip",
    "cards_issued",
    "credit_limit",
    "year_pin_last_changed",
    "acct_open_date_parsed",
    "expires_parsed"
).withColumn(
    "has_chip_flag",
    F.when(F.col("has_chip") == "YES", 1).otherwise(0)
).withColumn(
    "card_age_months",
    F.months_between(ref_date, F.col("acct_open_date_parsed"))
).withColumn(
    "months_until_expiry",
    F.months_between(F.col("expires_parsed"), ref_date)
).withColumn(
    "years_since_pin_change",
    F.year(ref_date) - F.col("year_pin_last_changed")
).drop("has_chip")

In [ ]:
card_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/card_features.parquet")

# transaction feature

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

txn_base = transactions \
    .withColumn(
        "date_str",
        F.concat_ws(
            "-",
            F.col("year").cast("string"),
            F.lpad(F.col("month").cast("string"), 2, "0"),
            F.lpad(F.col("day").cast("string"), 2, "0")
        )
    ) \
    .withColumn(
        "datetime",
        F.to_timestamp(F.concat_ws(" ", F.col("date_str"), F.col("time")), "yyyy-MM-dd HH:mm")
    ) \
    .withColumn("ts", F.unix_timestamp("datetime")) \
    .withColumn("abs_amount", F.abs(F.col("amount"))) \
    .withColumn("log_amount", F.log1p(F.abs(F.col("amount")))) \
    .withColumn("is_refund", F.when(F.col("amount") < 0, 1).otherwise(0)) \
    .withColumn("is_chip", F.when(F.col("use_chip") == "Chip Transaction", 1).otherwise(0)) \
    .withColumn("is_swipe", F.when(F.col("use_chip") == "Swipe Transaction", 1).otherwise(0)) \
    .withColumn("is_online", F.when(F.col("use_chip") == "Online Transaction", 1).otherwise(0)) \
    .withColumn("has_error", F.when(F.col("errors").isNotNull(), 1).otherwise(0)) \
    .withColumn("merchant_state_missing", F.when(F.col("merchant_state").isNull(), 1).otherwise(0)) \
    .withColumn("zip_missing", F.when(F.col("zip").isNull(), 1).otherwise(0)) \
    .withColumn("hour", F.hour("datetime")) \
    .withColumn("day_of_week", F.dayofweek("datetime")) \
    .withColumn("day_of_month", F.dayofmonth("datetime")) \
    .withColumn("week_of_year", F.weekofyear("datetime")) \
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("is_night", F.when((F.col("hour") <= 5) | (F.col("hour") >= 23), 1).otherwise(0)) \
    .withColumn("is_business_hour", F.when((F.col("hour") >= 9) & (F.col("hour") <= 17), 1).otherwise(0)) \
    .withColumn("hour_sin", F.sin(2 * 3.14159265359 * F.col("hour") / 24)) \
    .withColumn("hour_cos", F.cos(2 * 3.14159265359 * F.col("hour") / 24)) \
    .withColumn("dayofweek_sin", F.sin(2 * 3.14159265359 * F.col("day_of_week") / 7)) \
    .withColumn("dayofweek_cos", F.cos(2 * 3.14159265359 * F.col("day_of_week") / 7)) \
    .withColumn("is_fraud_flag", F.when(F.col("is_fraud") == "Yes", 1).otherwise(0))

In [ ]:
txn_base.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/txn_base.parquet")

# transaction_user feature

In [ ]:
txn_base = spark.read.parquet(f"{FEATURE_DATA_PATH}/txn_base.parquet")

In [ ]:
txn_base = spark.read.parquet(f"{FEATURE_DATA_PATH}/txn_base.parquet")

w_user_order = Window.partitionBy("user").orderBy("ts")
w_user_past = w_user_order.rowsBetween(Window.unboundedPreceding, -1)
w_user_prev_5 = w_user_order.rowsBetween(-5, -1)
w_user_prev_20 = w_user_order.rowsBetween(-20, -1)

w_user_1h = Window.partitionBy("user").orderBy("ts").rangeBetween(-3600, -1)
w_user_24h = Window.partitionBy("user").orderBy("ts").rangeBetween(-86400, -1)
w_user_7d = Window.partitionBy("user").orderBy("ts").rangeBetween(-604800, -1)

txn_user_features = txn_base.repartition("user") \
    .withColumn("user_prev_amount", F.lag("amount", 1).over(w_user_order)) \
    .withColumn("user_prev_ts", F.lag("ts", 1).over(w_user_order)) \
    .withColumn("user_seconds_since_prev_txn", F.col("ts") - F.col("user_prev_ts")) \
    .withColumn("user_amount_change_from_prev", F.col("amount") - F.col("user_prev_amount")) \
    .withColumn("user_txn_count_prev_5", F.count("*").over(w_user_prev_5)) \
    .withColumn("user_txn_count_prev_20", F.count("*").over(w_user_prev_20)) \
    .withColumn("user_avg_amount_prev_5", F.avg("amount").over(w_user_prev_5)) \
    .withColumn("user_avg_amount_prev_20", F.avg("amount").over(w_user_prev_20)) \
    .withColumn("user_std_amount_prev_20", F.stddev("amount").over(w_user_prev_20)) \
    .withColumn("user_avg_amount_past", F.avg("amount").over(w_user_past)) \
    .withColumn("user_std_amount_past", F.stddev("amount").over(w_user_past)) \
    .withColumn(
        "user_amount_zscore",
        (F.col("amount") - F.col("user_avg_amount_past")) / (F.col("user_std_amount_past") + F.lit(1e-6))
    ) \
    .withColumn("user_abs_amount_dev", F.abs(F.col("amount") - F.col("user_avg_amount_past"))) \
    .withColumn("user_refund_count_prev_20", F.sum("is_refund").over(w_user_prev_20)) \
    .withColumn("user_error_count_prev_20", F.sum("has_error").over(w_user_prev_20)) \
    .withColumn("user_online_count_prev_20", F.sum("is_online").over(w_user_prev_20)) \
    .withColumn("user_txn_count_last_1h", F.count("*").over(w_user_1h)) \
    .withColumn("user_txn_count_last_24h", F.count("*").over(w_user_24h)) \
    .withColumn("user_txn_count_last_7d", F.count("*").over(w_user_7d)) \
    .withColumn("user_avg_hour_past", F.avg("hour").over(w_user_past)) \
    .withColumn("user_hour_dev", F.abs(F.col("hour") - F.col("user_avg_hour_past")))

txn_user_features = txn_user_features.select(
    "user", "card", "datetime", "ts", "merchant_name",
    "user_prev_amount",
    "user_seconds_since_prev_txn",
    "user_amount_change_from_prev",
    "user_txn_count_prev_5",
    "user_txn_count_prev_20",
    "user_avg_amount_prev_5",
    "user_avg_amount_prev_20",
    "user_std_amount_prev_20",
    "user_avg_amount_past",
    "user_std_amount_past",
    "user_amount_zscore",
    "user_abs_amount_dev",
    "user_refund_count_prev_20",
    "user_error_count_prev_20",
    "user_online_count_prev_20",
    "user_txn_count_last_1h",
    "user_txn_count_last_24h",
    "user_txn_count_last_7d",
    "user_avg_hour_past",
    "user_hour_dev"
)

txn_user_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/txn_user_features.parquet")

In [ ]:
txn_user_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/txn_user_features.parquet")

# transaction_card feature

In [ ]:
txn_base = spark.read.parquet(f"{FEATURE_DATA_PATH}/txn_base.parquet")

w_card_order = Window.partitionBy("user", "card").orderBy("ts")
w_card_past = w_card_order.rowsBetween(Window.unboundedPreceding, -1)
w_card_prev_5 = w_card_order.rowsBetween(-5, -1)
w_card_prev_20 = w_card_order.rowsBetween(-20, -1)

w_card_1h = Window.partitionBy("user", "card").orderBy("ts").rangeBetween(-3600, -1)
w_card_24h = Window.partitionBy("user", "card").orderBy("ts").rangeBetween(-86400, -1)

txn_card_features = txn_base.repartition("user", "card") \
    .withColumn("card_prev_amount", F.lag("amount", 1).over(w_card_order)) \
    .withColumn("card_prev_ts", F.lag("ts", 1).over(w_card_order)) \
    .withColumn("card_seconds_since_prev_txn", F.col("ts") - F.col("card_prev_ts")) \
    .withColumn("card_txn_count_prev_5", F.count("*").over(w_card_prev_5)) \
    .withColumn("card_txn_count_prev_20", F.count("*").over(w_card_prev_20)) \
    .withColumn("card_avg_amount_prev_20", F.avg("amount").over(w_card_prev_20)) \
    .withColumn("card_std_amount_prev_20", F.stddev("amount").over(w_card_prev_20)) \
    .withColumn("card_avg_amount_past", F.avg("amount").over(w_card_past)) \
    .withColumn("card_std_amount_past", F.stddev("amount").over(w_card_past)) \
    .withColumn(
        "card_amount_zscore",
        (F.col("amount") - F.col("card_avg_amount_past")) / (F.col("card_std_amount_past") + F.lit(1e-6))
    ) \
    .withColumn("card_error_count_prev_20", F.sum("has_error").over(w_card_prev_20)) \
    .withColumn("card_online_count_prev_20", F.sum("is_online").over(w_card_prev_20)) \
    .withColumn("card_txn_count_last_1h", F.count("*").over(w_card_1h)) \
    .withColumn("card_txn_count_last_24h", F.count("*").over(w_card_24h))

txn_card_features = txn_card_features.select(
    "user", "card", "datetime", "ts", "merchant_name",
    "card_prev_amount",
    "card_seconds_since_prev_txn",
    "card_txn_count_prev_5",
    "card_txn_count_prev_20",
    "card_avg_amount_prev_20",
    "card_std_amount_prev_20",
    "card_avg_amount_past",
    "card_std_amount_past",
    "card_amount_zscore",
    "card_error_count_prev_20",
    "card_online_count_prev_20",
    "card_txn_count_last_1h",
    "card_txn_count_last_24h"
)

txn_card_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/txn_card_features.parquet")

In [ ]:
txn_card_features.write.mode("overwrite").parquet(f"{FEATURE_DATA_PATH}/txn_card_features.parquet")